### **Mount to the Drive**



In [36]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2'

%cd $base_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2


In [37]:
left_img_path = f'{base_dir}/data/kitti_sample/sequences/00/image_2/000000.png'
right_img_path = f'{base_dir}/data/kitti_sample/sequences/00/image_3/000000.png'
calib_path = f'{base_dir}/data/kitti_sample/sequences/00/calib.txt'

In [38]:
# Run the command below if you need
!pip install -q -r requirements/3d_reconstruction.txt

In [39]:
import cv2 as cv
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from src.visualization import visualize_stereo_reconstruction

### **Calibration**



In [40]:
with open(calib_path) as calib:
  calib_lines = calib.readlines()

projection_2 = np.array([float(x) for x in calib_lines[2][4:].split()])
projection_2.resize(3, 4)

projection_3 = np.array([float(x) for x in calib_lines[3][4:].split()])
projection_3.resize(3, 4)

f = projection_2[0, 0]
baseline = -(projection_3[0, 3] - projection_2[0, 3])/f
cx = projection_2[0, 2]
cy = projection_2[1, 2]

In [41]:
reprojection_matrix = np.array([
    [1,                 0,                0,                -cx               ],
    [0,                 1,                0,                -cy               ],
    [0,                 0,                0,                f                 ],
    [0,                 0,                -1/baseline,      0                 ]
    ])

In [42]:
left_img = cv.imread(left_img_path)
left_img = cv.cvtColor(left_img, cv.COLOR_BGR2RGB)

left_img_gray = cv.imread(left_img_path, cv.IMREAD_GRAYSCALE)
right_img_gray = cv.imread(right_img_path, cv.IMREAD_GRAYSCALE)

### **SGBM Algorithm: Disparity Map**



In [43]:
window_size = 3
min_disp = 0
num_disp = 128

In [44]:
stereo = cv.StereoSGBM_create(minDisparity = min_disp,
        numDisparities = num_disp,
        blockSize = 7,
        P1 = 8*3*window_size**2,
        P2 = 32*3*window_size**2,
        disp12MaxDiff = 1,
        uniquenessRatio = 10,
        speckleWindowSize = 100,
        speckleRange = 2,
        mode = cv.STEREO_SGBM_MODE_SGBM_3WAY
)

disparity_SGBM = stereo.compute(left_img_gray, right_img_gray)
disparity_SGBM = disparity_SGBM.astype(np.float32) / 16.0

### **3D Point Cloud Generation**



In [45]:
h, w, c = left_img.shape
reconstruction = cv.reprojectImageTo3D(disparity_SGBM, reprojection_matrix)

In [46]:
mask = disparity_SGBM>0

left_img_masked = left_img[mask]
reconstruction_masked = reconstruction[mask]

In [47]:
points = reconstruction_masked.copy()
colours = left_img_masked.copy()

points[:, 2] = -points[:, 2]

In [48]:
def export_ply(filename, points, colors):
    ply_header = f"""ply
      format ascii 1.0
      element vertex {len(points)}
      property float x
      property float y
      property float z
      property uchar red
      property uchar green
      property uchar blue
      end_header
      """
    try:
        with open(filename, 'w') as f:
            f.write(ply_header)
            for p, c in zip(points, colors):
                f.write(f"{p[0]:.4f} {p[1]:.4f} {p[2]:.4f} {int(c[0])} {int(c[1])} {int(c[2])}\n")

        print(f"Point cloud is successfully saved: {filename}")

    except Exception as e:
        print(f"Error: {e}")

export_ply(f"{base_dir}/outputs/kitti_3d_model.ply", points, colours)

Point cloud is successfully saved: /content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2/outputs/kitti_3d_model.ply


### **Interactive 3D Visualization**



In [49]:
fig = visualize_stereo_reconstruction(points, colours)
fig.show(renderer="colab")

Output hidden; open in https://colab.research.google.com to view.